# Step 3: Data cleaning

Inspect quality issues, document the chosen handling for each, and write the final cleaned data to `../data/cleaned/orders_cleaned.csv`. Do not overwrite the source CSV.

In [55]:
import pandas as pd

df = pd.read_csv("../data/raw/quick_commerce_orders_simulated.csv")
print(df.shape)

(100800, 16)


In [56]:
# Check 1 :
print("Duplicate full rows: ",df.duplicated().sum())
print("Duplicate Order Ids : ",df["Order ID"].duplicated().sum())

Duplicate full rows:  767
Duplicate Order Ids :  800


Duplicates: remove them, don't rename them.
Renaming would invent a fake order. One real order must appear once, so we delete the extra copy.

A real order ID should appear only once. If it appears twice, something is wrong.

In [57]:
# Check 2:
print(df["Platform"].value_counts())

Platform
Blinkit             44622
Swiggy Instamart    35148
JioMart             19820
BLINKIT               196
Blinkit               184
blinkit               180
Instamart             140
Swiggy                133
swiggy instamart      131
JIOMART                87
Jio Mart               81
jiomart                78
Name: count, dtype: int64


In [58]:
print(df["City"].value_counts(dropna=False))


City
Mumbai       24014
Delhi        23857
Bengaluru    21705
Hyderabad    11834
Chennai       8958
Pune          8920
NaN           1512
Name: count, dtype: int64


In [59]:
print(df["Product Category"].value_counts())

Product Category
Grocery                27945
Fruits & Vegetables    20217
Dairy                  17317
Snacks                 14122
Beverages              12165
Personal Care           9034
Name: count, dtype: int64


Look at the min and max rows. Can a delivery take 0 or negative minutes? Can an order be worth ₹0? Does anything look absurdly large for a grocery order?

In [60]:
# Check3 :

df[["Order Value (INR)", "Delivery Time (Minutes)","Delivery Distance (km)", "Delivery Delay (Minutes)"]].describe()


,Order Value (INR),Delivery Time (Minutes),Delivery Distance (km),Delivery Delay (Minutes)
count,100800.000000,100800.000000,100800.000000,100800.000000
mean,399.326141,15.970833,1.985409,0.897887
std,822.417410,8.931709,1.325826,2.998500
min,-199.000000,-5.000000,0.300000,0.000000
25%,205.000000,11.000000,1.000000,0.000000
50%,311.000000,15.000000,1.700000,0.000000
75%,485.000000,19.000000,2.700000,0.000000
max,99999.000000,999.000000,9.000000,34.000000


here we see that some Extrem value or say  neagtive values : min and max 

If a delivery time says -5 minutes, nobody knows the true number, and a guess becomes made-up data. The safer options are:

Mark the bad value as missing (NaN).
Then either drop the row or fill it with something defensible, such as the median for that platform.

In [61]:
# chake rating :

print(df["Service Rating"].value_counts(dropna=False).sort_index())

Service Rating
1.0      386
2.0     2715
3.0     8634
4.0    31495
5.0    35113
NaN    22457
Name: count, dtype: int64


In [62]:
# 767 duplicate full rows but 800 duplicate Order IDs
# which colum are differ in 33 pair 

dupes = df[df["Order ID"].duplicated(keep=False)]

differs = dupes.groupby("Order ID").nunique(dropna=False)
print((differs >1 ).sum())

Customer ID                  0
Platform                     0
City                        15
Order Date & Time            0
Product Category             0
Order Value (INR)            1
Payment Method              17
Delivery Distance (km)       0
Promised SLA (Minutes)       0
Delivery Time (Minutes)      0
Delivery Delay (Minutes)     0
Delayed                      0
Service Rating               0
Customer Feedback            0
Refund Requested             0
dtype: int64


In [63]:
# the exact platform labels
print((df["Platform"].value_counts()))

Platform
Blinkit             44622
Swiggy Instamart    35148
JioMart             19820
BLINKIT               196
Blinkit               184
blinkit               180
Instamart             140
Swiggy                133
swiggy instamart      131
JIOMART                87
Jio Mart               81
jiomart                78
Name: count, dtype: int64


In [64]:
print(df["Delivery Time (Minutes)"].sort_values().head(10).tolist())
print(df["Delivery Time (Minutes)"].sort_values().tail(10).tolist())
print(df["Order Value (INR)"].sort_values().head(10).tolist())
print(df["Order Value (INR)"].sort_values().tail(10).tolist())

[-5, -5, -5, -5, -5, -5, -5, -5, -5, -5]
[450, 450, 450, 450, 450, 450, 450, 450, 999, 999]
[-199, -199, -199, -199, -199, -199, -199, -199, -199, -199]
[3763, 3900, 4000, 4000, 99999, 99999, 99999, 99999, 99999, 99999]


Delivery Time: -5, 0, 450, 999. A delivery can't take 0 or negative minutes. And 450 minutes (7.5 hours) is not a quick-commerce delivery.
Order Value: -199, 0, 99999. The 99999 looks like a placeholder or system error.
Don't treat everything large as an error. ₹3,763 to ₹4,000 is a plausible large grocery basket, so keep those.

In [65]:
df_clean = df.copy()

In [66]:
platform_map = {
    "blinkit": "Blinkit",
    "swiggy": "Swiggy Instamart",
    "swiggy instamart": "Swiggy Instamart",
    "instamart": "Swiggy Instamart",
    "jiomart": "JioMart",
    "jio mart": "JioMart",
}

df_clean["Platform"]= (df_clean["Platform"].str.strip().str.lower().map(platform_map))
print(df_clean["Platform"].value_counts(dropna=False))

Platform
Blinkit             45182
Swiggy Instamart    35552
JioMart             20066
Name: count, dtype: int64


In [67]:
# remove impossible values
bad_time = (df_clean["Delivery Time (Minutes)"] <= 0) | (df_clean["Delivery Time (Minutes)"] > 180)
bad_value = (df_clean["Order Value (INR)"]<=0) | (df_clean["Order Value (INR)"] > 10000)

df_clean = df_clean[~(bad_value | bad_time)]
print(df_clean.shape)

(100735, 16)


In [68]:
# remove duplicates, keeping the most complete copy

df_clean["n_missing"] = df_clean.isna().sum(axis=1)
df_clean = (df_clean.sort_values("n_missing").drop_duplicates(subset="Order ID",keep="first").drop(columns="n_missing"))

print(df_clean.shape)
print("Duplicate Order IDs left:", df_clean["Order ID"].duplicated().sum())


(99936, 16)
Duplicate Order IDs left: 0


In [69]:
# handle missing values
df_clean["Customer Feedback"] = df_clean["Customer Feedback"].fillna("No Comments")
df_clean["City"] = df_clean["City"].fillna("Unknown")
df_clean["Payment Method"] = df_clean["Payment Method"].fillna("Unknown")

print(df_clean.isna().sum())

Order ID                        0
Customer ID                     0
Platform                        0
City                            0
Order Date & Time               0
Product Category                0
Order Value (INR)               0
Payment Method                  0
Delivery Distance (km)          0
Promised SLA (Minutes)          0
Delivery Time (Minutes)         0
Delivery Delay (Minutes)        0
Delayed                         0
Service Rating              22260
Customer Feedback               0
Refund Requested                0
dtype: int64


Only Service Rating should still show missing values.

In [70]:
df_clean["Order Date & Time"] = pd.to_datetime(df_clean["Order Date & Time"])
df_clean["Order Date"] = df_clean["Order Date & Time"].dt.date
df_clean["Hour"] = df_clean["Order Date & Time"].dt.hour
df_clean["Weekday"] = df_clean["Order Date & Time"].dt.day_name()
df_clean["Is Weekend"] = df_clean["Order Date & Time"].dt.dayofweek >=5 
print(df_clean.dtypes["Order Date & Time"])
print(df_clean[["Order Date & Time", "Hour", "Weekday", "Is Weekend"]].head())

datetime64[us]
        Order Date & Time  Hour   Weekday  Is Weekend
34781 2025-02-01 10:59:05    10  Saturday        True
34780 2025-02-01 10:58:42    10  Saturday        True
34779 2025-02-01 10:58:29    10  Saturday        True
34778 2025-02-01 10:58:19    10  Saturday        True
34777 2025-02-01 10:57:16    10  Saturday        True


In [71]:
print((df_clean["Delivery Time (Minutes)"] <= 0).sum())
print((df_clean["Delivery Time (Minutes)"] > 180).sum())
print((df_clean["Order Value (INR)"] <= 0).sum())
print((df_clean["Order Value (INR)"] > 10000).sum())

0
0
0
0


In [72]:
df_clean = df_clean.sort_values("Order Date & Time").reset_index(drop=True)

print(df_clean.shape)
print(df_clean["Platform"].nunique(), "platforms")
print("Duplicate IDs:", df_clean["Order ID"].duplicated().sum())
print(df_clean["Service Rating"].dropna().between(1, 5).all())
print(df_clean[["City", "Payment Method"]].isin(["Unknown"]).sum())

df_clean.to_csv("../data/cleaned/orders_cleaned.csv", index=False)

(99936, 20)
3 platforms
Duplicate IDs: 0
True
City              1496
Payment Method     990
dtype: int64
